In [3]:
import os
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
from llavaguard_config import local_data_dir

In [ ]:
# split into train/test/validation sets
from itertools import product
from sklearn.model_selection import train_test_split
import pandas as pd

template_version = 'v1'

file_path = f'{local_data_dir}/PEGI-LlavaGuard-DS/all_data.json'
ds_out = f"{local_data_dir}/PEGI-LlavaGuard-DS/{template_version}"

df = pd.read_json(file_path, orient="records")

train_split, val_split, test_split = [], [], []
categories = df["category"].unique()
ratings    = df["PEGI-rating"].unique()

for category, pegi in product(categories, ratings):
    subset = df[(df["category"] == category) & (df["PEGI-rating"] == pegi)]
    n = len(subset)
    if n < 2:
        continue
    test_size, val_size = 8, 1
    if pegi == "PEGI18":
        test_size = 10 # better one?
        val_size  = 2 

    test_size = max(1, min(test_size, n-2))
    val_size = max(0, min(val_size, n - test_size))

    print(f'[{category} x {pegi}] total={n}, test={test_size}, val={val_size}, train={n-test_size-val_size}')

    train_val, test = train_test_split(
        subset, test_size=test_size, random_state=42, shuffle=True
    )
    if val_size > 0:
        train, val = train_test_split(
            train_val, test_size=val_size, random_state=42, shuffle=True
        )
    else:
        train, val = train_val, subset.iloc[[]]  # empty val DataFrame

    train_split.append(train)
    val_split.append(val)
    test_split.append(test)


train_df = pd.concat(train_split, ignore_index=True)
val_df   = pd.concat(val_split,   ignore_index=True)
test_df  = pd.concat(test_split,  ignore_index=True)

print(f"\nFinal sizes → train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")


train_df.to_json(f"{ds_out}/PEGI-train.json", orient="records", indent=2)
val_df.to_json(  f"{ds_out}/PEGI-val.json",   orient="records", indent=2)
test_df.to_json( f"{ds_out}/PEGI-test.json",  orient="records", indent=2)


In [ ]:
# split the data
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

sns.set_style("ticks")
#data = np.random.normal(size=(20, 6)) + np.arange(6) / 2
#sns.boxplot(data=data);
#sns.despine()
template_version = "v1"
#json_path = '/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/PEGI-LlavaGuard-DS/v1/PEGI-test.json'
json_path = f'{local_data_dir}/PEGI-Guard-DS/{template_version}/PEGI-test.json'

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)
df = pd.DataFrame(data)


df["category_sort"] = df["category"].apply(lambda x: int(re.match(r"^\d+", x).group()))
df = df.sort_values("category_sort")
ordered_categories = df["category"].drop_duplicates()


df["PEGI-rating"] = df["PEGI-rating"].astype(str)
df["PEGI-rating"] = df["PEGI-rating"].replace("-1", "Not Rated")
pegi_order = ["Not Rated", "3", "7", "12", "16", "18"]
df["PEGI-rating"] = pd.Categorical(df["PEGI-rating"], categories=pegi_order, ordered=True)


# plot
sns.set_style("whitegrid")
sns.set_theme(rc={'figure.figsize':(20,40)})
g = sns.displot(
    data=df,
    x="category",
    y="PEGI-rating",
    cbar=True,
    height=7,
    aspect=2 
)

for ax in g.axes.flat:
    for label in ax.get_xticklabels():
        label.set_rotation(45)
        label.set_horizontalalignment('right')
    ax.set_yticks(pegi_order)  # use defined order directly
    ax.invert_yaxis()          # flip y-axis so "Not Rated" is at the bottom



g.fig.subplots_adjust(bottom=0.25)